# BUAD 5722 — Group 13 | Crime Data Pipeline

**Run `BigDataFinal_Census.ipynb` first** to produce `state_crime_census_2022.csv`.

**Data sources:**
- FBI NIBRS 2022 — ICPSR Study 38925 (11.2M incident records)
- US Census ACS 5-year 2022 — state-level socioeconomic variables

In [ ]:
# Phase 1: Install all project dependencies
!pip install pyspark pandas geopandas plotly scikit-learn xgboost requests anthropic -q

In [ ]:
# Import all libraries used across all phases
import zipfile, time
import pandas as pd
import geopandas as gpd
import plotly
import requests
import anthropic
from pathlib import Path
from sklearn import preprocessing
from google.colab import drive
import xgboost as xgb

In [ ]:
# Config: all paths and keys in one place
CENSUS_API_KEY = 'e3a26cc02893ae40c47a3c8b52a02c7e4c8d5d72'

# Shared Drive — all teammates access the same data files
DRIVE_FOLDER = Path('/content/drive/Shareddrives/MSBA Group 13/Big Data /BUAD5722_Crime_Project')

# Local paths (recreated each Colab session)
NIBRS_DIR  = Path('/content/crime_project_data/nibrs_raw')
OUTPUT_DIR = Path('/content/crime_project_data/clean')
NIBRS_FILE = NIBRS_DIR / 'ICPSR_38925/DS0002/38925-0002-Data.tsv'

In [ ]:
# Mount Google Drive and create local directories
drive.mount('/content/drive')

for d in [NIBRS_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Drive mounted. Ready.')

In [ ]:
# Phase 2: Extract NIBRS ZIP from shared Drive
# The ZIP is stored in the shared Drive so all teammates use the same source file
# Skips extraction if the file already exists (safe to re-run)

if NIBRS_FILE.exists():
    print('Already extracted.')
else:
    zip_path = list(DRIVE_FOLDER.glob('*.zip'))[0]
    print(f'Found: {zip_path.name}  ({zip_path.stat().st_size/1e6:.0f} MB)')
    print('Extracting... (this may take a minute)')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(NIBRS_DIR)
    print('Done.')

# List extracted files
print('\nFiles in NIBRS directory:')
for f in sorted(NIBRS_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')

## NIBRS file guide

| File | Contents |
|------|----------|
| `38925-0001-Data.tsv` | Batch header — agency metadata (4 MB) |
| `38925-0002-Data.tsv` | **Administrative file — one row per incident (737 MB) use this** |
| `38925-0003-Data.tsv` | Offense-level file (10 GB) |
| `38925-0004-Data.tsv` | Victim-level file (8.9 GB) |
| `38925-0005-Data.tsv` | Arrestee-level file (5.9 GB) |
| `38925-0006-Data.tsv` | Offender-level file (11 GB) |

In [ ]:
# Preview NIBRS administrative file — check raw column names
# Raw columns use ICPSR V-codes; renamed to readable labels in the next cell

df_preview = pd.read_csv(NIBRS_FILE, sep='\t', nrows=5, low_memory=False)
print(f'{len(df_preview.columns)} columns: {list(df_preview.columns)}')
display(df_preview)

In [ ]:
# Column name mapping: V-codes to readable labels
# Source: ICPSR DS0002 codebook (38925-0002-Codebook-ICPSR.pdf)
# Setup files (SAS/SPSS) were not in the ZIP, so mapping was done manually

COL_NAMES = {
    'V1001': 'ori',                   # Agency identifier (Originating Agency Identifier)
    'V1002': 'incident_number',       # Unique incident number
    'FIPS_STATE': 'fips_state',       # State FIPS code — join key for Census data
    'V1003': 'agency_state',          # State abbreviation + agency ORI
    'V1004': 'population_group',      # Agency population group code
    'V1005': 'division',              # Census division code
    'V1006': 'region',                # Census region code
    'V1007': 'agency_indicator',      # Agency type indicator
    'V1008': 'core_city',             # Core city indicator
    'V1009': 'covered_by_ori',        # Covered-by ORI (reports under another agency)
    'V1010': 'num_months_reported',   # Months agency submitted data this year
    'V1011': 'incident_date',         # Date of incident (DD-MON-YYYY)
    'V1012': 'report_date_indicator', # Whether date is incident or report date
    'V1013': 'incident_hour',         # Hour of incident (0-23)
    'V1014': 'cleared_exceptionally', # Clearance flag (N = not cleared)
    'V1015': 'num_offenses',          # Number of offenses in this incident
    'V1016': 'num_victims',           # Number of victims in this incident
}

# Preview with renamed columns
df_preview = pd.read_csv(NIBRS_FILE, sep='\t', nrows=10, low_memory=False)
df_preview = df_preview.rename(columns=COL_NAMES)
print(f'Columns: {list(df_preview.columns)}')
display(df_preview)